# 1 · Fundamentos de las caminatas cuánticas discretas

Una **caminata cuántica en tiempo discreto** (DTQW) evoluciona un *walker* sobre un grafo alternando dos operaciones:

1. una **moneda** (coin) que mezcla los grados de libertad internos, y
2. un **desplazamiento** (shift) condicionado a ese estado interno.

A diferencia de la caminata clásica difusiva, la interferencia hace que la distribución se separe en dos *cuernos* y que el walker avance de forma **balística** (`sigma ~ t`). Este notebook recorre los ejemplos básicos:

- distribución de la moneda de Hadamard en la recta,
- cómo la moneda de rotación ajusta la velocidad de grupo, y
- animaciones en la recta y en el ciclo.

In [ ]:
# Setup común. Ejecuta esta celda primero.
%matplotlib inline
import sys, os
try:
    import zitterwalk  # noqa: F401
except ModuleNotFoundError:
    # notebook dentro de examples/: añade la raíz del proyecto al path
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image

from zitterwalk import Graph, Walker, DiscreteTimeWalk
from zitterwalk import viz


## 1.1 · Caminata de Hadamard en la recta

Reproduce la distribución de dos cuernos característica de la DTQW y la compara con la campana difusiva de la caminata clásica. (`examples/line_hadamard.py`)

In [ ]:
n = 201                 # recta larga para no tocar los bordes
center = n // 2
steps = 80

g = Graph.line(n)
walk = DiscreteTimeWalk(g, coin="hadamard")

# estado inicial simétrico (|left> + i|right>) / sqrt(2)
w = Walker.at_node(g, center, coin_state=[1, 1j])

states = walk.run(w, steps)
p_final = walk.probabilities(states[-1])

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))
positions = np.arange(n) - center

# 1) distribución final, quedándonos con la paridad alcanzable
parity = steps % 2
mask = (np.arange(n) % 2) == (center + parity) % 2
ax1.plot(positions[mask], p_final[mask], color="crimson")
ax1.set_title(f"DTQW Hadamard, t = {steps}")
ax1.set_xlabel("posición"); ax1.set_ylabel("probabilidad")

# 2) evolución temporal (frente balístico)
viz.plot_evolution(walk, states, ax=ax2)

# 3) comparación con difusión clásica (sigma ~ t/sqrt(2))
sigma = steps / np.sqrt(2)
gauss = np.exp(-positions ** 2 / (2 * sigma ** 2)); gauss /= gauss.sum()
ax3.plot(positions[mask], p_final[mask], color="crimson", label="cuántica")
ax3.plot(positions, gauss, color="navy", ls="--", label="clásica (difusión)")
ax3.set_title("Cuántica vs clásica"); ax3.set_xlabel("posición"); ax3.legend()

fig.tight_layout()

mean = (positions * p_final).sum()
std = np.sqrt(((positions - mean) ** 2 * p_final).sum())
print(f"desviación estándar tras {steps} pasos: {std:.1f}  (~ balística, O(t))")

## 1.2 · Ajustando la moneda: velocidad de grupo

La moneda de rotación de un parámetro `C(theta) = [[cos, sin], [sin, -cos]]` controla cuán rápido se dispersa la caminata: los picos se sitúan en `±cos(theta)·t`. (`examples/coin_dispersion.py`)

In [ ]:
from zitterwalk import rotation

n = 301
center = n // 2
steps = 100
g = Graph.line(n)
w = Walker.at_node(g, center, coin_state=[1, 1j])
pos = np.arange(n) - center

thetas = [0.15 * np.pi, 0.25 * np.pi, 0.35 * np.pi, 0.45 * np.pi]
fig, ax = plt.subplots(figsize=(9, 5))

for theta in thetas:
    walk = DiscreteTimeWalk(g, coin=rotation(theta))
    final = walk.step(w, steps)
    p = walk.probabilities(final)
    mask = (np.arange(n) % 2) == (center + steps) % 2
    ax.plot(pos[mask], p[mask],
            label=fr"$\theta$ = {theta/np.pi:.2f}$\pi$, "
                  fr"$v = \cos\theta$ = {np.cos(theta):.2f}")
    ax.axvline(np.cos(theta) * steps, color="0.85", lw=0.8, zorder=0)
    ax.axvline(-np.cos(theta) * steps, color="0.85", lw=0.8, zorder=0)

ax.set_xlabel("posición"); ax.set_ylabel("probabilidad")
ax.set_title(f"Moneda de rotación C($\\theta$), t = {steps}\n"
             "(líneas grises: picos previstos en $\\pm\\cos\\theta \\cdot t$)")
ax.legend()
fig.tight_layout()

## 1.3 · Animación en la recta

Anima la probabilidad por nodo de una caminata de Hadamard en la recta y la guarda como GIF. (`examples/line_animation.py`)

In [ ]:
n = 151
center = n // 2
t_max = 70

g = Graph.line(n)
walk = DiscreteTimeWalk(g, coin="hadamard")
w = Walker.at_node(g, center, coin_state=[1, 1j])
states = walk.run(w, t_max)

viz.animate(walk, states, save_path="line_walk.gif", kind="line", fps=10)
plt.close("all")   # evita que se muestre también el frame estático
Image("line_walk.gif")

## 1.4 · Animación en el ciclo

El mismo estilo pero sobre un ciclo pequeño: el walker da la vuelta e interfiere consigo mismo. (`examples/cycle_animation.py`)

In [ ]:
n = 24           # ciclo pequeño
start = 0
t_max = 60

g = Graph.cycle(n)
walk = DiscreteTimeWalk(g, coin="hadamard")
w = Walker.at_node(g, start, coin_state=[1, 1j])
states = walk.run(w, t_max)

viz.animate(walk, states, save_path="cycle_walk.gif", kind="line", fps=10)
plt.close("all")
Image("cycle_walk.gif")

## 1.5 · Ciclo con nodos coloreados

Los nodos se disponen en anillo y su color se anima según su probabilidad. (`examples/cycle_nodes_animation.py`)

In [ ]:
n = 24           # ciclo pequeño para ver bien los nodos
start = 0
t_max = 60

g = Graph.cycle(n)
walk = DiscreteTimeWalk(g, coin="hadamard")
w = Walker.at_node(g, start, coin_state=[1, 1j])
states = walk.run(w, t_max)

viz.animate(walk, states, save_path="cycle_nodes.gif", kind="graph",
            fps=10, node_size=650)
plt.close("all")
Image("cycle_nodes.gif")